# Resnet50 classification pytorch

Original experiment from [Neural-Networks-and-Deep-Learning-Using-Pytorch-and-Tensor-Flow](https://github.com/Muhammad-Huzifa/Neural-Networks-and-Deep-Learning-Using-Pytorch-and-Tensor-Flow/blob/dfa17b2b298631b10e5904b2f1c39ad64d424bc1/Deep%20Neural%20Networks/CNN/Classification%20Using%203%20Classes%20Resnet50/Classification%20of%20Balls%20Cars%20and%20Cone%20using%20Resnet50.ipynb). Run from a fresh kernel. See [dataset and environment instructions](../../docs/DATASETS.md).

Framework training, downloaded datasets, and custom image inputs require the dependencies and resources described in the collection README. This migration has checked syntax and paths; it has not repeated every training experiment.

In [ ]:
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "requirements/base.txt").is_file())
TABULAR = ROOT / "data/tabular"
IMAGES = ROOT / "data/images"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)


In [ ]:
import torch
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, random_split
import os

# Define dataset root directory
data_dir = str(IMAGES / "image_classification")

# Define transformations (resize, normalize, etc.)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load the dataset without predefined train/test split
dataset = datasets.ImageFolder(root=data_dir, transform=transform)

# Get class names
class_names = dataset.classes
print("Classes:", class_names)


In [ ]:
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, random_split

# Define dataset transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset (without train/test split)
dataset = datasets.ImageFolder(root=data_dir, transform=transform)

# Split 80% Training / 20% Validation
train_ratio = 0.8
train_size = int(train_ratio * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Create DataLoaders
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"✅ Dataset Loaded! Training: {len(train_dataset)}, Validation: {len(val_dataset)}")


In [ ]:
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load ResNet-50 (pretrained)
model = models.resnet50(pretrained=True)

# Modify the final layer for 3 classes
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 3)

# Move model to device
model = model.to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)


In [ ]:
EPOCHS = 10

# Store loss & accuracy
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []



In [ ]:
# Check if train_loader has data
data_iter = iter(train_loader)
images, labels = next(data_iter)

print(f"Batch Size: {images.shape}, Labels: {labels}")


In [ ]:
print(f"Using Device: {device}")
print(f"Model Device: {next(model.parameters()).device}")

# Move model to device if not already
model.to(device)


In [ ]:
import os
print(os.listdir(str(IMAGES / "image_classification")))


In [ ]:
print(f"Number of batches in train_loader: {len(train_loader)}")


In [ ]:
# Print weights before update
print("Before update:", model.fc.weight[0][0].item())

optimizer.zero_grad()
outputs = model(images.to(device))
loss = criterion(outputs, labels.to(device))
loss.backward()
optimizer.step()

# Print weights after update
print("After update:", model.fc.weight[0][0].item())


In [ ]:
print(f"Model in training mode: {model.training}")  # Should print True


In [ ]:
for epoch in range(EPOCHS):
    # Training
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)



In [ ]:
# Validation
model.eval()
running_loss, correct, total = 0.0, 0, 0

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

val_loss = running_loss / len(val_loader)
val_acc = 100 * correct / total
val_losses.append(val_loss)
val_accuracies.append(val_acc)

print(f"Epoch [{epoch+1}/{EPOCHS}], Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")


In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(train_losses) + 1)  # Ensure x-axis matches

plt.figure(figsize=(10, 5))

# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(epochs, train_losses, label='Training Loss', marker='o')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training Loss per Epoch')
plt.legend()

# Plot Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs, train_accuracies, label='Training Accuracy', marker='o', color='green')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.title('Training Accuracy per Epoch')
plt.legend()

plt.show()
